In [25]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
import torch.nn.functional as F
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import SubsetRandomSampler
from torch.utils.data import sampler
import numpy as np
import matplotlib.colors as mcolors

In [26]:
# Hyperparameters
RANDOM_SEED = 49      ## Random number seed
LEARNING_RATE = 0.0002  ## learning rate
BATCH_SIZE = 256      ##Batch size for stochastic learning
NUM_EPOCHS = 50       ##Number of epochs
NUM_CLASSES = 10      ##number of classes in image
device = torch.device("cuda:0")

In [27]:
## Data loader and generator
def get_dataloaders_mnist(batch_size, num_workers=0,
                          train_transforms=None, test_transforms=None):
  """
    Function: To create dataloader and datagenerator from dataset
    Input:
      batch_size: Size of individual batch of data
      num_workers: number of parallel workers
      train_transforms: Transformations applied to training dataset
      test_transforms: Transformations applied to testing dataset

  """
  ## Add transform to training input
  if train_transforms is None:
      train_transforms = transforms.Compose([
          transforms.ToTensor()
      ])
  ## Add transform to test input
  if test_transforms is None:
      test_transforms = transforms.Compose([
          transforms.ToTensor()
      ])
  ## Import the training dataset
  train_dataset = datasets.MNIST(root='data',
                                train=True,
            transform=train_transforms,download=True)
  ## Import the validation dataset
  valid_dataset = datasets.MNIST(root='data',
                                  train=True,
                                  transform=test_transforms)
  ## Import the test dataset
  test_dataset = datasets.MNIST(root='data',
                                train=False,
                                transform=test_transforms)
  ## Convert the training dataset into iterative dataloader
  train_loader = DataLoader(dataset=train_dataset,
                            batch_size=batch_size,
                            num_workers=num_workers,
                            shuffle=True)
  ## Convert the validation dataset into iterative dataloader
  valid_loader = DataLoader(dataset=valid_dataset,
                            batch_size=batch_size,
                            num_workers=num_workers,
                            shuffle=False)
  ## Convert the test dataset into iterative dataloader
  test_loader = DataLoader(dataset=test_dataset,
                            batch_size=batch_size,
                            num_workers=num_workers,
                            shuffle=False)
  return train_loader, valid_loader, test_loader
train_loader, valid_loader, test_loader = get_dataloaders_mnist(batch_size=BATCH_SIZE,
    num_workers=2)


In [28]:
# Checking the dataset
print('Training Set:\n')
for images, labels in train_loader:
    print('Image batch dimensions:', images.size())
    print('Image label dimensions:', labels.size())
    print(labels[:10])
    break

Training Set:

Image batch dimensions: torch.Size([256, 1, 28, 28])
Image label dimensions: torch.Size([256])
tensor([8, 0, 8, 4, 6, 5, 9, 5, 1, 4])


In [29]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        ##Encoder model
        self.encoder = nn.Sequential(
                ##IC = 1 OC = 32
                nn.Conv2d(1, 32, stride=(1, 1), kernel_size=(3, 3),     padding=1),
                nn.LeakyReLU(0.01),
                ##IC = 32 OC =64
                nn.Conv2d(32, 64, stride=(2, 2), kernel_size=(3, 3), padding=1),
                nn.LeakyReLU(0.01),
                ##IC =64 OC =64
                nn.Conv2d(64, 64, stride=(2, 2), kernel_size=(3, 3), padding=1),
                nn.LeakyReLU(0.01),
                ##IC =64 OC =64
                nn.Conv2d(64, 64, stride=(1, 1), kernel_size=(3, 3), padding=1),
                nn.Flatten(),
                ##Flatten to linear
                nn.Linear(3136, 2)
        )

        ##Decoder model
        self.decoder = nn.Sequential(
                ## Start with the linear layer
                torch.nn.Linear(2, 3136),
                ##Reshape to 64x7x7
                nn.Unflatten(-1, (64, 7, 7)),
                ##Stack of transpose conv2d layers
                nn.ConvTranspose2d(64, 64, stride=(1, 1), kernel_size=(3, 3), padding=1),
                nn.LeakyReLU(0.01),
                nn.ConvTranspose2d(64, 64, stride=(2, 2), kernel_size=(3, 3), padding=1),
                nn.LeakyReLU(0.01),
                nn.ConvTranspose2d(64, 32, stride=(2, 2), kernel_size=(3, 3), padding=0),
                nn.LeakyReLU(0.01),
                nn.ConvTranspose2d(32, 1, stride=(1, 1), kernel_size=(3, 3), padding=0),
                nn.ZeroPad2d((-1,0,-1,0)),  # 1x29x29 -> 1x28x28
                nn.Sigmoid()
                )
    ## Forward run
    def forward(self, x):
      x = self.encoder(x)
      x = self.decoder(x)
      return x
model = AutoEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [30]:
def train_autoencoder(num_epochs, model, optimizer,
                         train_loader, loss_fn=None,
                         logging_interval=100,
                         skip_epoch_stats=False,
                         save_model=None):
  log_dict = {'train_loss_per_batch': [],
                'train_loss_per_epoch': []}
  if loss_fn is None:
      loss_fn = F.mse_loss
  start_time = time.time()
  for epoch in range(num_epochs):
      model.train()
      for batch_idx, (features, _) in enumerate(train_loader):
          # FORWARD AND BACK PROP
          features = features.to(device)
          logits = model(features)
          loss = loss_fn(logits, features)
          optimizer.zero_grad()
          loss.backward()
          # UPDATE MODEL PARAMETERS
          optimizer.step()
          # LOGGING
          log_dict['train_loss_per_batch'].append(loss.item())
          if not batch_idx % logging_interval:
              print('Epoch: %03d/%03d | Batch %04d/%04d | Loss: %.4f'
                    % (epoch+1, num_epochs, batch_idx,
                        len(train_loader), loss))
      if not skip_epoch_stats:
          model.eval()
          with torch.set_grad_enabled(False):  # save memory during inference
              train_loss = compute_epoch_loss_autoencoder(
                  model, train_loader, loss_fn)
              print('***Epoch: %03d/%03d | Loss: %.3f' % (
                    epoch+1, num_epochs, train_loss))

          log_dict['train_loss_per_epoch'].append(train_loss.item())
  print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
  print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))
  if save_model is not None:
      torch.save(model.state_dict(), save_model)
  return log_dict

log_dict = train_autoencoder(num_epochs=NUM_EPOCHS, model=model,
                                optimizer=optimizer,
                                train_loader=train_loader,
                                skip_epoch_stats=True,
                                logging_interval=250)

Epoch: 001/050 | Batch 0000/0235 | Loss: 0.2233
Epoch: 002/050 | Batch 0000/0235 | Loss: 0.0644
Epoch: 003/050 | Batch 0000/0235 | Loss: 0.0543
Epoch: 004/050 | Batch 0000/0235 | Loss: 0.0523
Epoch: 005/050 | Batch 0000/0235 | Loss: 0.0515
Epoch: 006/050 | Batch 0000/0235 | Loss: 0.0503
Epoch: 007/050 | Batch 0000/0235 | Loss: 0.0510
Epoch: 008/050 | Batch 0000/0235 | Loss: 0.0481
Epoch: 009/050 | Batch 0000/0235 | Loss: 0.0467
Epoch: 010/050 | Batch 0000/0235 | Loss: 0.0464
Epoch: 011/050 | Batch 0000/0235 | Loss: 0.0476
Epoch: 012/050 | Batch 0000/0235 | Loss: 0.0461
Epoch: 013/050 | Batch 0000/0235 | Loss: 0.0483
Epoch: 014/050 | Batch 0000/0235 | Loss: 0.0469
Epoch: 015/050 | Batch 0000/0235 | Loss: 0.0455
Epoch: 016/050 | Batch 0000/0235 | Loss: 0.0449
Epoch: 017/050 | Batch 0000/0235 | Loss: 0.0446
Epoch: 018/050 | Batch 0000/0235 | Loss: 0.0431
Epoch: 019/050 | Batch 0000/0235 | Loss: 0.0452
Epoch: 020/050 | Batch 0000/0235 | Loss: 0.0433
Epoch: 021/050 | Batch 0000/0235 | Loss:

In [31]:
print(log_dict)

{'train_loss_per_batch': [0.2233223170042038, 0.22074562311172485, 0.21914339065551758, 0.2172398567199707, 0.21523027122020721, 0.2130812555551529, 0.21073703467845917, 0.20874431729316711, 0.20615939795970917, 0.20369650423526764, 0.2001994401216507, 0.1971701830625534, 0.19298231601715088, 0.18837611377239227, 0.18294315040111542, 0.17652040719985962, 0.16789354383945465, 0.15993471443653107, 0.14974525570869446, 0.14215093851089478, 0.13199356198310852, 0.12510518729686737, 0.12582001090049744, 0.12182369083166122, 0.11799352616071701, 0.11814416944980621, 0.11805906891822815, 0.11311980336904526, 0.10854450613260269, 0.10949115455150604, 0.10959676653146744, 0.1028829962015152, 0.10297033190727234, 0.09487795829772949, 0.09566131979227066, 0.0960976779460907, 0.09389793127775192, 0.08889784663915634, 0.08894956111907959, 0.08762719482183456, 0.08467761427164078, 0.08447688072919846, 0.07941244542598724, 0.08301977068185806, 0.08004341274499893, 0.07622276246547699, 0.0782188251614